In [13]:
import os
print(os.getcwd())

/mnt/shared-storage-gpfs2/beam-gpfs02/yulang/master/nemo_cellflow/preprocessing/arcinstitute/datasets/State_Replogle_Filtered


In [2]:
# 读取TOML文件
from pathlib import Path
import tomllib
few_shot_name = "additive"
toml_path = Path(f"few_shot/{few_shot_name}_split/{few_shot_name}.toml")  # 改成你的路径

with toml_path.open("rb") as f:
    cfg = tomllib.load(f)
    
cfg["fewshot"].keys()

dict_keys(['norman.k562'])

In [6]:
# celltype and drug in val and test set
from itertools import product
val_cartesian_fewshot_list = []
test_cartesian_fewshot_list = []

for x in cfg["fewshot"].keys():
    cell_name = x.split(".")[-1]
    val_drug_names = cfg["fewshot"][x]["val"]
    val_cartesian_fewshot = val_drug_names
    test_drug_names = cfg["fewshot"][x]["test"]
    test_cartesian_fewshot = test_drug_names
    val_cartesian_fewshot_list.extend(val_cartesian_fewshot)
    test_cartesian_fewshot_list.extend(test_cartesian_fewshot)



val_cartesian_fewshot_list = [(x.split('_')[0], x.split('_')[1]) for x in val_cartesian_fewshot_list]
test_cartesian_fewshot_list = [(x.split('_')[0], x.split('_')[1]) for x in test_cartesian_fewshot_list]


print(val_cartesian_fewshot_list[:10])
print(test_cartesian_fewshot_list[:10])
print(len(val_cartesian_fewshot_list))
print(len(test_cartesian_fewshot_list))

[('MAP2K6', 'ELMSAN1'), ('PTPN12', 'ZBTB25'), ('FOXA3', 'FOXF1'), ('KIF18B', 'KIF2C'), ('POU3F2', 'CBFA2T3'), ('CEBPE', 'SPI1'), ('FOSB', 'PTPN12'), ('MAP2K6', 'IKZF3'), ('SGK1', 'S1PR2'), ('KLF1', 'TGFBR2')]
[('MAP2K6', 'ELMSAN1'), ('PTPN12', 'ZBTB25'), ('FOXA3', 'FOXF1'), ('KIF18B', 'KIF2C'), ('POU3F2', 'CBFA2T3'), ('CEBPE', 'SPI1'), ('FOSB', 'PTPN12'), ('MAP2K6', 'IKZF3'), ('SGK1', 'S1PR2'), ('KLF1', 'TGFBR2')]
14
14


In [ ]:
import pickle
with open("pert_sample_list.pkl", "rb") as f:
    pert_data = pickle.load(f)
    
from tqdm import tqdm
train_sample_list = []
val_sample_list = []
test_sample_list = []
for x in tqdm(pert_data):



    
    cell_drug_product = (x["cartesian_key"][0], x["cartesian_key"][1])
    if cell_drug_product in val_cartesian_fewshot_list:
        val_sample_list.append(x)
        test_sample_list.append(x)
    elif cell_drug_product in test_cartesian_fewshot_list:
        test_sample_list.append(x)
    else:
        train_sample_list.append(x)

len(val_sample_list)

100%|██████████| 893/893 [00:00<00:00, 1329610.75it/s]


33

In [12]:
# 分别写入lmdb
import lmdb
import pickle
from tqdm import tqdm
import os

def dump_list_to_lmdb(
    data_list,
    lmdb_path: str,
    map_size_bytes: int,   # 必须足够大（比如原始数据大小 * 1.2~2）
    protocol: int = pickle.HIGHEST_PROTOCOL,
):
    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    with env.begin(write=True) as txn:
        # 存长度
        txn.put(b"__len__", str(len(data_list)).encode("utf-8"))

    # 分批写：避免一个 txn 太大
    batch = 10_000
    for start in tqdm(range(0, len(data_list), batch), desc="writing lmdb"):
        end = min(start + batch, len(data_list))
        with env.begin(write=True) as txn:
            for i in range(start, end):
                k = str(i).encode("utf-8")
                v = pickle.dumps(data_list[i], protocol=protocol)
                txn.put(k, v)

    env.sync()
    env.close()
def fewshot_lmdb_dir(split="train"):
    base_dir = f"few_shot/{few_shot_name}_split"
    lmdb_dir = os.path.join(base_dir, f"norman_{split}_{few_shot_name}")
    return lmdb_dir

lmdb_dict = {fewshot_lmdb_dir("train"): train_sample_list, fewshot_lmdb_dir("val"): val_sample_list, fewshot_lmdb_dir("test"): test_sample_list}
for key, data_list in lmdb_dict.items():
    dump_list_to_lmdb(data_list, key, map_size_bytes=180 * (1 << 30))
    
    


writing lmdb: 100%|██████████| 1/1 [00:00<00:00, 124.56it/s]


In [13]:
import pickle
with open("control_dict.pkl", "rb") as f:
    control_data = pickle.load(f)
list(control_data.keys())

[('control', 'None')]

In [14]:
import pickle
with open("control_dict.pkl", "rb") as f:
    control_data = pickle.load(f)
    
    
# 字符串，tuple互相转换脚本
import ast
test_tuple = list(control_data.keys())[0]
test_str = str(test_tuple)
print(len(test_tuple))
print(len(test_str))
print(len(ast.literal_eval(test_str)))

def dump_dict_to_lmdb(
    data_dict,
    lmdb_path: str,
    map_size_bytes: int,
    protocol: int = pickle.HIGHEST_PROTOCOL,
):

    env = lmdb.open(
        lmdb_path,
        map_size=map_size_bytes,
        subdir=True,
        create=True,
        lock=True,
        readahead=False,
        max_dbs=1,
    )

    keys = list(data_dict.keys())

    with env.begin(write=True) as txn:
        txn.put(b"__len__", str(len(keys)).encode("utf-8"))
        txn.put(b"__keys__", pickle.dumps(keys, protocol=protocol))

        for k_str, obj in tqdm(data_dict.items()):
            k = str(k_str).encode("utf-8")
            v = pickle.dumps(obj, protocol=protocol)
            txn.put(k, v)

    env.sync()
    env.close()

# 写入lmdb
dump_dict_to_lmdb(control_data, fewshot_lmdb_dir("control"), map_size_bytes=180 * (1 << 30))

2
19
2


100%|██████████| 1/1 [00:00<00:00, 91.43it/s]
